# Default packages and General helper functions

In [ ]:
# Data types
from typing import List, Optional, Union
from enum import Enum
from numpy.typing import NDArray

# Visualization
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# Data handling
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# UNet
import torch
from torch import nn
from torch.optim import Adam
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# RL Environment
import gymnasium
from gymnasium import Env
from gymnasium.spaces import Box, Discrete, Dict, Tuple
from gymnasium import spaces

# RL training
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.policies import ActorCriticPolicy
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.evaluation import evaluate_policy

# Saving and loading
import pickle

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"{device=}")


In [ ]:
# Observation space
class Tile(Enum):
  OCEAN = 0
  SUBMARINE = 1
  ISLAND = 2
  TRAVELLED = 3

# Action space
class Action(Enum):
  UP = 0
  DOWN = 1
  LEFT = 2
  RIGHT = 3
  NOTHING = 4
  RESURFACE = 5

_vectorize_action = {
    # Going from row 1 to 0 is UP
    Action.UP.value:      np.array([ -1 ,  0 ]),
    # Going from row 0 to 1 is DOWN
    Action.DOWN.value:    np.array([  1 ,  0 ]),
    # Going from column 1 to 0 is LEFT
    Action.LEFT.value:    np.array([  0 , -1 ]),
    # Going from column 0 to 1 is RIGHT
    Action.RIGHT.value:   np.array([  0 ,  1 ]),
    Action.NOTHING.value: np.array([  0 ,  0 ]),
}

board_numpy = np.array(
    [
        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 0, 2, 0, 0,  0, 2, 0, 0, 0,  0, 0, 2, 2, 0],
        [0, 0, 2, 0, 0,  0, 0, 0, 2, 0,  0, 0, 2, 0, 0],
        [0, 0, 0, 0, 0,  0, 0, 0, 2, 0,  0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],

        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 2, 0, 2, 0,  0, 2, 0, 2, 0,  0, 0, 0, 0, 0],
        [0, 2, 0, 2, 0,  0, 2, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 0, 0, 2, 0,  0, 0, 2, 0, 0,  0, 2, 2, 2, 0],
        [0, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],

        [0, 0, 0, 2, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
        [0, 0, 2, 0, 0,  0, 0, 2, 0, 0,  0, 2, 0, 0, 0],
        [2, 0, 0, 0, 0,  0, 0, 0, 0, 0,  0, 0, 2, 0, 0],
        [0, 0, 2, 0, 0,  0, 2, 0, 2, 0,  0, 0, 0, 2, 0],
        [0, 0, 0, 2, 0,  0, 0, 0, 0, 0,  0, 0, 0, 0, 0],
    ]
)

# All code for UNet

<!-- Template
<details>
<summary>&#10004; [Text] </summary>
[Explanation]
</details>
 -->

**UNet Clarity**

<details>
<summary>&#10004; Standardize inputs for all UNet functions </summary>
Provide data type hints to UNet functions <br>
Add check functions that make sure proper dimensions are put in
</details>
<details>
<summary> &#10004; Implement resurfacing </summary>
Idea: Project resurfacing onto separate input <br>
Explanation: Each time the sub resurfaces, it gets recorded which section it resurfaces. This section is projected in a separate input.  The sequence recorded is started from empty again. Each path begins with no section or with a section
</details>
<details>
<summary>&#10005; Adept the new UNet code from BEP phase 3 to BEP phase 4 </summary>
Adjust the standardize masks with masks to pans <br>
Adjust the train_model
</details>

In [ ]:
# Standardize UNet-helper functions
# Accepts only 2D — (15, 15)
# Accepts only numpy

grid_size = 15

def add_islands(ax):
    for x in range(grid_size):
        for y in range(grid_size):
            if board_numpy[x, y] == 2:
                circle = Circle((y, x), 0.5, fill=True, color="green", alpha=1)
                ax.add_patch(circle)

def sections_to_masks(sections : List[Optional[int]]) -> NDArray[np.int_]:
    masks = np.zeros((len(sections), grid_size, grid_size), dtype=np.int_)

    for index, section in enumerate(sections):
        if section == None or section == -1:
            continue;
        for x_area in range(5):
            for y_area in range(5):
                x, y = x_area + (section // 3) * 5, y_area + (section % 3) * 5
                masks[index, x, y] = 1
    return masks

def actions_to_paths(actions : List[Union[str, List[int]]]) -> List[List[np.ndarray]]:
    if type(actions[0]) == list:
        sequences = [[_vectorize_action[direction] for direction in action] for action in actions]
    if type(actions[0]) == str:
        sequences = [[_vectorize_action[int(number_string)] for number_string in string.strip().split(" ")] for string in actions]
    sequences = [[np.array([0, 0])] + nested_list for nested_list in sequences]

    for sequence in sequences:
        for i, vector in enumerate(sequence[1:]):
            sequence[i + 1] = sequence[i] + vector

            if np.any(sequence[i + 1] < 0):
                sequence[:i + 2] = sequence[:i + 2] - vector
    return sequences

def paths_to_masks(paths : List[List[np.ndarray]]) -> NDArray[np.int_]:
    paths_masked = np.zeros((len(paths), grid_size, grid_size))
    for index, path in enumerate(paths):
        for i, pos in enumerate(path):
            x, y = pos[0], pos[1]
            if i == len(path) - 1:
                paths_masked[index, x, y] = Tile.SUBMARINE.value
            else:
                paths_masked[index, x, y] = Tile.TRAVELLED.value

    return paths_masked

# def standardize_masks(masks : NDArray[np.int_]) -> NDArray[np.int_]:
#     paths_masked_standardized = masks.copy()
#     # For each mask ...
#     for i in range(masks.shape[0]):
#         mask = paths_masked_standardized[i]
#         # Get rid of empty rows and columns
#         pan, _ = mask_to_pan(mask)
#         # Add those rows and columns back to the right and bottom
#         pad_rows, pad_cols = 15 - pan.shape[0], 15 - pan.shape[1]
#         padded = np.pad(
#             pan,
#             pad_width=((0, pad_rows), (0, pad_cols)),
#             mode='constant',
#             constant_values=Tile.OCEAN.value
#         )
#         # Store result
#         paths_masked_standardized[i] = padded
#     return paths_masked_standardized

def standardize_masks(masks : NDArray[np.int_]) -> NDArray[np.int_]:
    filters, _ = masks_to_pans(masks)

    # Find how many rows and columns to pad for each filter
    pad_rows = [15 - filter.shape[0] for filter in filters]
    pad_cols = [15 - filter.shape[1] for filter in filters]
    # As filters are homogeneous, we need to pad for each filter separately without numpy :(
    standardized_masks = [np.pad(
        filter,
        pad_width=((0, pad_row), (0, pad_col)),
        mode='constant',
        constant_values=0
    ) for filter, pad_row, pad_col in zip(filters, pad_rows, pad_cols)]
    # Make it a numpy array as it is standardized to (N, 15, 15)
    return np.array(standardized_masks)

def check_2D(array : NDArray[np.int_]) -> bool:
    if len(array.shape) == 2:
        return True
    raise ValueError(f"Only 2D arrays are accepted {array.shape=}")

def mask_to_pan(mask : NDArray[np.int_]) -> tuple[NDArray[np.int_], int]:
    # Only accept 2D arrays
    check_2D(mask)
    grid_size = mask.shape[0]
    pan = mask.copy()
    # Get rid of empty rows and columns
    pan = pan[np.abs(pan.sum(axis=1)) != 0]
    pan = pan.T[np.abs(pan.sum(axis=0)) != 0].T
    # Get the position of submarine in pan
    indices = np.where(pan == Tile.SUBMARINE.value)
    (x, y) = indices
    # Return pan and project submarine pan position to original grid
    flattened_grid_position = x * grid_size + y
    return (pan, flattened_grid_position)

def masks_to_pans(masks : NDArray[np.int_]) -> tuple[List[NDArray[np.int_]], List[int]]:
    if len(masks.shape) != 3:
        raise ValueError(f"Expected masks shape is 3D, got {masks.shape=}")
    if type(masks) != np.ndarray:
        raise ValueError(f"Expected masks type is np.ndarray, got {type(masks)=}")

    # Find the empty rows and columns
    X = masks.sum(axis=1) != 0
    Y = masks.sum(axis=2) != 0
    # Get rid of the empty rows and columns
    filters = [mask[y_mask].T[x_mask].T for mask, x_mask, y_mask in zip(masks, X, Y)]
    # Find flat sub positions in the original grid
    indices_x, indices_y = np.where(masks == 2)[1:]
    sub_positions = indices_x * 15 + indices_y
    return filters, sub_positions

def deterministic_sub_finding(mask : NDArray[np.int_], board : NDArray[np.int_]) -> NDArray[np.float64]:
    # Only accept 2D arrays
    check_2D(mask)
    check_2D(board)

    # Get the pan for panning
    pan, sub_pos = mask_to_pan(mask)

    # Get the shape of the pan and the board to calculate the output shape of convolution
    x_shape, y_shape = pan.shape
    X_shape, Y_shape = board.shape
    x_out, y_out = X_shape - x_shape + 1, Y_shape - y_shape + 1

    # Change to torch tensors and add batch and channel dimensions required for convolution
    board_torch = torch.tensor(board).view(1, 1, X_shape, Y_shape).float()
    pan_torch = torch.tensor(pan).view(1, 1, x_shape, y_shape).float()

    # Perform the Convolution
    output = F.conv2d(board_torch, pan_torch, stride=1, padding=0)

    # Check values of each position, as islands are worth 0 and ocean is worth 1, the maximum value of convolution is the sum of pan
    output = torch.where(output == pan_torch.sum(), 1, 0)

    # Normalize the output to get a probability distribution
    output = output / output.sum()

    # Offset the position of possible submarine locations to the original grid
    # Convolution output is always smaller than the original grid
    flattened_output = output.flatten().cpu().detach().numpy()
    flat_matrix = np.zeros((X_shape * Y_shape))
    for position in range(x_out * y_out):
        value = flattened_output[position]
        if value > 0:
            x_smaller, y_smaller = position // y_out, position % y_out
            adjusted_position = x_smaller * grid_size + y_smaller
            flat_matrix[adjusted_position + sub_pos] = value

    # Reshape the flat matrix back to the original grid shape
    matrix = flat_matrix.reshape(X_shape, Y_shape)
    return matrix

In [ ]:
def check_4D(array : NDArray[np.int_]) -> bool:
    if len(array.shape) == 4:
        return True
    raise ValueError(f"Only 4D arrays are accepted {array.shape=}")

class SequenceMapped(Dataset):
    def __init__(self, mapped_paths, final_positions, board, resurfaces):
        self.mapped_paths = torch.tensor(mapped_paths).unsqueeze(1).float()
        # Numpy of (X, 2, 15, 15). The two channels contain begin resurface and end resurface
        self.resurfaces = torch.tensor(resurfaces).unsqueeze(1).float()
        self.final_positions = torch.tensor(np.array(final_positions)).unsqueeze(1).long()
        self.board = torch.tensor(board).float().unsqueeze(0).repeat(self.mapped_paths.shape[0], 1, 1).unsqueeze(1)

        check_4D(self.mapped_paths)
        check_4D(self.resurfaces)
        check_4D(self.board)
        check_2D(self.final_positions)

    def __len__(self):
        return self.final_positions.size()[0]

    def __getitem__(self, index):   
        return self.mapped_paths[index], self.final_positions[index], self.board[index], self.resurfaces[index]
    
class SequenceMappedTest(SequenceMapped):
    def __init__(self, mapped_paths, final_positions, board, resurfaces, paths):
        # List of strings to list of list of numbers
        super().__init__(mapped_paths, final_positions, board, resurfaces)
        self.paths = paths

    def __getitem__(self, index):
        return self.mapped_paths[index], self.final_positions[index], self.board[index], self.paths[index], index,self.resurfaces[index]

In [ ]:
# UNet architecture inspired by https://arxiv.org/abs/1505.04597

class UNet(nn.Module):
    def __init__(self, num_channels, input_channel = 2, grid_size=15, num_directions= len(Action), bias=True):
        super().__init__()

        self.grid_size = grid_size

        self.UPSAMPLE = nn.Upsample(scale_factor=2)
        self.DOWNSAMPLE = nn.MaxPool2d(2, 2)
        self.INPUT = nn.Sequential(
            nn.Conv2d(input_channel, num_channels, kernel_size=3, padding=1, stride=1, bias=bias),
            nn.ReLU(),
            nn.Conv2d(num_channels,  num_channels, kernel_size=3, padding=1, stride=1, bias=bias),
            nn.ReLU(),
            nn.Conv2d(num_channels,  num_channels, kernel_size=3, padding=1, stride=1, bias=bias),
        )
        self.DOWN1 = nn.Sequential(
            nn.Conv2d(num_channels,     num_channels * 2, kernel_size=3, padding=1, stride=1, bias=bias, groups=2),
            nn.ReLU(),
            nn.Conv2d(num_channels * 2, num_channels * 2, kernel_size=3, padding=1, stride=1, bias=bias, groups=2),
            nn.ReLU(),
            nn.Conv2d(num_channels * 2, num_channels * 2, kernel_size=3, padding=1, stride=1, bias=bias, groups=2),
        )
        self.BOTTOM = nn.Sequential(
            nn.Conv2d(num_channels * 2, num_channels * 4, kernel_size=3, padding=1, stride=1, bias=bias, groups=4),
            nn.ReLU(),
            nn.Conv2d(num_channels * 4, num_channels * 4, kernel_size=3, padding=1, stride=1, bias=bias, groups=4),
            nn.ReLU(),
            nn.Conv2d(num_channels * 4, num_channels * 4, kernel_size=3, padding=1, stride=1, bias=bias, groups=4),
        )
        self.UP1 = nn.Sequential(
            nn.Conv2d(num_channels * 2 + num_channels * 4 , num_channels * 2, kernel_size=3, padding=1, stride=1, bias=bias, groups=2),
            nn.ReLU(),
            nn.Conv2d(num_channels * 2                    , num_channels * 2, kernel_size=3, padding=1, stride=1, bias=bias, groups=2),
            nn.ReLU(),
            nn.Conv2d(num_channels * 2                    , num_channels * 2, kernel_size=3, padding=1, stride=1, bias=bias, groups=2),
        )
        self.OUTPUT = nn.Sequential(
            nn.Conv2d(num_channels * 1 + num_channels * 2 , num_channels * 1, kernel_size=3, padding=1, stride=1, bias=bias),
            nn.ReLU(),
            nn.Conv2d(num_channels * 1                    , num_channels * 1, kernel_size=3, padding=1, stride=1, bias=bias),
            nn.ReLU(),
            nn.Conv2d(num_channels * 1                    , 1               , kernel_size=3, padding=1, stride=1, bias=bias),
            nn.BatchNorm2d(1),
            nn.ReLU()
        )
        self.softmax = nn.LogSoftmax(dim=1)
        self.flatten = nn.Flatten()

    def forward(self, mapping : torch.Tensor, board : torch.Tensor, resurface : torch.Tensor) -> torch.Tensor:
        # Pad such that it is 16x16 for the downsampling and upsampling to work properly
        board_padded = F.pad(board, (0, 1, 0, 1))
        mapping_padded = F.pad(mapping, (0, 1, 0, 1))
        resurface_padded = F.pad(resurface, (0, 1, 0, 1))

        # Stack X_padded and board_padded
        stacked = torch.hstack((board_padded, mapping_padded, resurface_padded)).float()
        Layer1Encoder = self.INPUT(stacked)
        Layer1EncoderDownsample = self.DOWNSAMPLE(Layer1Encoder)
        Layer2Encoder = self.DOWN1(Layer1EncoderDownsample)
        Layer2EncoderDownsample = self.DOWNSAMPLE(Layer2Encoder)
        BOTTLENECK = self.BOTTOM(Layer2EncoderDownsample)

        BOTTLENECKUpsample = self.UPSAMPLE(BOTTLENECK)
        stacked = torch.hstack((Layer2Encoder, BOTTLENECKUpsample))
        Layer2Decoder = self.UP1(stacked)

        Layer2DecoderUpsample = self.UPSAMPLE(Layer2Decoder)
        stacked = torch.hstack((Layer1Encoder, Layer2DecoderUpsample))
        i2o_padded = self.OUTPUT(stacked)

        # Remove the padding
        i2o = i2o_padded[:, :, :self.grid_size, :self.grid_size]
        # Mask using the board to exploit information about islands
        i2o = i2o * board
        # CrossEntropyLoss applies softmax internally, make sure input is 1D
        i2o = self.flatten(i2o)

        return i2o

In [ ]:
# Train UNet model

def train_model(
    train_data : SequenceMapped,
    test_data : SequenceMapped,
    model : UNet,
    loss_fn : torch.nn.Module,
    epochs=10,
    lr=0.01,
    print_every=1,
    visualize_images=100,
    batch_size=32,
    device="cpu"
):
    loss_dict = {"train": [], "test": []}

    # We use a `DataLoader` to get batching for free!
    train_data_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, generator=torch.Generator(device=device))
    test_data_loader = DataLoader(test_data, batch_size=1, shuffle=True, generator=torch.Generator(device=device))
    optimizer = Adam(model.parameters(), lr=lr)

    # Print header.
    print(f"Epoch    Train RMSRE      Test RMSRE")

    for epoch in range(epochs):
        model.train()
        epoch_loss_sum = 0
        for mapped_path, final_position, board, resurface in train_data_loader:
            # Reset optimizer gradients.
            optimizer.zero_grad()
            output = model(mapped_path, board, resurface)
            # Compute the loss
            loss = loss_fn(output, final_position.squeeze(1))
            epoch_loss_sum += loss.item()
            # Compute gradients according to newly computed loss.
            loss.backward()
            # Update the model parameters.
            optimizer.step()

        loss_dict["train"].append(epoch_loss_sum / len(train_data_loader))

        with torch.no_grad():
            model.eval()

            vis_count = 0
            for masked_path_torch, final_position, board, path, index, resurface in test_data_loader:
                output_torch = model(masked_path_torch, board, resurface)
                output_numpy = output_torch.cpu().detach().numpy().squeeze().reshape(15, 15)
                masked_path_numpy = masked_path_torch.cpu().detach().numpy().squeeze().reshape(15, 15)
                board_numpy = board.cpu().detach().numpy().squeeze().reshape(15, 15)

                fig, ax = plt.subplots(1, 2, figsize = (8, 8))
                ax[0].imshow(output_numpy)
                deterministic_output = deterministic_sub_finding(masked_path_numpy, board_numpy)
                ax[1].imshow(deterministic_output)

                add_islands(ax[0])
                add_islands(ax[1])

                length_path = len(path)
                for index, pos in enumerate(path):
                    pos = pos.cpu().detach().flatten()
                    y, x = pos[0], pos[1]
                    alpha = ((index+1) / length_path) ** 2
                    alpha = 1
                    circle = Circle((x, y), 0.2, fill=True, color="red", alpha=alpha)
                    ax[0].add_patch(circle)
                    circle = Circle((x, y), 0.2, fill=True, color="red", alpha=alpha)
                    ax[1].add_patch(circle)

                index = np.argmax(output_numpy)
                x = index % 15
                y = index // 15
                circle = Circle((x, y), 0.2, fill=True, color="black", alpha=1)
                ax[0].add_patch(circle)
                circle = Circle((x, y), 0.2, fill=True, color="black", alpha=1)
                ax[1].add_patch(circle)

                for a in ax.flat:
                    a.axis('off')

                ax[0].set_title("UNet Heatmap Output")
                ax[1].set_title("Deterministic Heatmap Output")

                plt.show()

                if vis_count > visualize_images:
                    break;
                else:
                    vis_count += 1


            test_pred = model(test_data.mapped_paths, test_data.board, test_data.resurfaces)
            test_loss = loss_fn(test_pred, test_data.final_positions.squeeze(1))
            loss_dict["test"].append(test_loss.item())

            if (epoch + 1) % print_every == 0:
                train_pred = model(train_data.mapped_paths, train_data.board, train_data.resurfaces)
                print(
                    f"{epoch+1: <7}  {loss_fn(train_pred, train_data.final_positions.squeeze(1)).item(): <12.6e}  {loss_fn(test_pred, test_data.final_positions.squeeze(1)).item(): .6e}"
                )

    model.eval()
    return model, loss_dict

def plot_losses(train, test):
    plt.semilogy(train, label="train", marker=".", lw=2)
    plt.semilogy(test, label="test", marker=".", lw=2)
    plt.xlabel("Epoch")
    plt.ylabel("RMSRE")
    plt.legend()
    plt.show()

In [ ]:
# Data needed for UNet
board_UNet = np.where(board_numpy == 2, 0, 1)

In [ ]:
# Test the code

with open("intermediate_sequences.pkl", 'rb') as file:
  intermediate_sequences = pickle.load(file)
with open("intermediate_paths.pkl", 'rb') as file:
  intermediate_paths_duplicates = pickle.load(file)

indexes = [[pos[0] * 15 + pos[1] for pos in path] for path in intermediate_paths_duplicates]
indexes = [np.array(path) for path in indexes]
for i, path in enumerate(indexes):
  if len(np.unique(path)) < len(path):
    unique_values, indices = np.unique(path, return_index=True)
    ordered_unique = unique_values[np.argsort(indices)]
    indexes[i] = ordered_unique

intermediate_paths = [[np.array([index // 15, index % 15]) for index in list(path)] for path in indexes]

paths_sequences_standardized = actions_to_paths(intermediate_sequences)
for index, (sequence, path) in enumerate(zip(paths_sequences_standardized, intermediate_paths)):
    if len(sequence) != len(path):
        paths_sequences_standardized[index] = path[:len(path)]

In [ ]:
def data_processing_pipeline(
        sequences_movement : List[Union[str, List[int]]], 
        sections_resurfaces : List[Optional[int]], 
        intermediate_paths : List[List[NDArray[np.int_]]],
        random_seed = 42) -> tuple[SequenceMapped, SequenceMappedTest]:
    masks_current = paths_to_masks(sequences_movement)
    paths_mapped_standardized = standardize_masks(masks_current)
    final_positions = [path[-1][0] * 15 + path[-1][1] for path in intermediate_paths]
    resurfaces_UNet = sections_to_masks(sections_resurfaces)

    train_inputs, test_inputs, train_labels, test_labels = train_test_split(
        paths_mapped_standardized, final_positions, test_size=0.2, random_state=random_seed, shuffle=False
    )
    train_resurfaces_UNet, test_resurfaces_UNet, _, test_paths = train_test_split(
        resurfaces_UNet, intermediate_paths, test_size=0.2, random_state=random_seed, shuffle=False
    )

    train_data_current = SequenceMapped(train_inputs, train_labels, board_UNet, train_resurfaces_UNet)
    test_data_current = SequenceMappedTest(test_inputs, test_labels, board_UNet, test_resurfaces_UNet, test_paths)
    return train_data_current, test_data_current

train_data_current, test_data_current = data_processing_pipeline(paths_sequences_standardized, [-1] * len(paths_sequences_standardized), intermediate_paths)

In [ ]:
initial_UNet_model, loss_dict = train_model(
    train_data_current,
    test_data_current,
    UNet(num_channels=32, input_channel=3),
    nn.CrossEntropyLoss(),
    epochs=2,
    lr=0.01,
    print_every=1,
    visualize_images=1,
    batch_size=32,
    device=device
)

# All code for RL Agent 2 and 4


<!-- Template
<details>
<summary>&#10005; [Text] </summary>
[Explanation]
</details>
 -->


**RL Clarity**

<details>
<summary> &#10004; Implement the coordinates send from torpedo </summary>
&#10004; Add another observation key in the observation Dict for both agents <br>
&#10004; Add the reset of torpedo coordinates <br>
&#10004; Add countdown to fire torpedo <br>
&#10004; Add UNet prediction when countdown reaches 0 <br>
</details>
&#10004; Implement resurfacing <br>
&#10005; Code RL agent 2 and 4 and train them <br> 
&#10005; Implement the dance between RL and UNet. Simulate X games of RL, let UNet learn back and forth <br> 
&#10004; Fix the data collection of both sequence of actions and positions recorded (thus no recording of data when true action is nothing) <br>
<details>
<summary>&#10005; Implement proper data collection via callback  </summary>
&#10004; Collect sequence data with list[str] <br>
&#10004; Collect path data <br>
&#10004; Collect data about starting section after resurfacing. Not after it is done running<br>
</details>

## General environment setup

In [ ]:
class ActionLoggerCallback(BaseCallback):
    def __init__(self, verbose=0):
      super().__init__(verbose)
      self.actions_taken : str = ""
      self.whole_actions_taken : str = ""
      self.path_taken : list[NDArray[np.int_]] = []
      self.whole_path_taken : list[NDArray[np.int_]] = []
      self.resurface_section : Optional[int] = None

      self.sequences : list[str]= []
      self.whole_sequences : list[str] = []
      self.paths : list[list[NDArray[np.int_]]] = []
      self.whole_paths : list[list[NDArray[np.int_]]] = []
      self.sections : list[Optional[int]] = []

    def _on_step(self) -> bool:
      # Log the action taken
      info = self.locals.get('infos', [{}])[0]
      true_action = info.get('true_action')
      position = info.get("agent_position")
      record = info.get("record")
      done = self.locals.get('dones', [False])[0]
      
      if true_action is not None:
        self.actions_taken += str(true_action) + " "
        self.whole_actions_taken += str(true_action) + " "
        self.path_taken.append(position)
        self.whole_path_taken.append(position)

      # Either record when resurfacing, submarine dies, or episode ends
      if record == True:
        if len(self.path_taken) > 10:
          self.sequences.append(self.actions_taken.strip())
          self.paths.append(self.path_taken)
          self.sections.append(self.resurface_section)
          self.whole_sequences.append(self.whole_actions_taken.strip())
          self.whole_paths.append(self.whole_path_taken)

        # Reset for next record
        self.actions_taken = ""
        self.path_taken = []
        self.resurface_section = info.get("section")

      if done == True:
        # Reset for next episode
        self.whole_actions_taken = ""
        self.whole_path_taken = []

      return True

In [ ]:
fig, ax = plt.subplots()

def CycleModel(model, callback, className, cycles = 100):
  mean, std = [], []

  for _ in range(cycles):
    print(f"Number {_} of {cycles}")
    model.learn(total_timesteps=2048, callback=callback)
    model.get_env().reset()
    m, s = evaluate_policy(model, model.get_env(), n_eval_episodes=25, deterministic=True)
    print(f"Deterministic mean: {m} and std: {s}")
    mean.append(m)
    std.append(s)

  mean = np.array(mean)
  std = np.array(std)
  ax.plot(range(cycles), mean, label=className)
  ax.fill_between(range(cycles), mean - std, mean+std, alpha=0.2)

In [ ]:
class Torpedo_Counter():
    def __init__(self, seed = 42, low = 3, high = 8, max_fires = 10):
        self.low = low
        self.high = high
        self.rng = np.random.default_rng(seed)
        self.cooldown_time = 0
        self.generate_cooldown()
        self.fired = 0
        self.max_fires = max_fires

    def generate_cooldown(self):
        self.cooldown_time = self.rng.integers(low=self.low, high=self.high)

    def tick(self):
        self.cooldown_time -= 1

    def reset(self):
        self.fired = 0
        self.generate_cooldown()

    def can_fire(self):
        if self.fired >= self.max_fires:
            return False
        if self.cooldown_time <= 0:
            self.generate_cooldown()
            self.fired += 1
            return True
        else:
            self.tick()
            return False

In [ ]:
class SimpleCaptainSonarEnv(Env):

  metadata = {"render_modes": ["human"], "render_fps": 30}

  def __init__(self, board, UNet, torpedo_counter : Torpedo_Counter, max_length = 50):
    super().__init__()
    self.agent               = np.array([0, 0], dtype=int)
    self.repairable_movement = np.array([0] * len(Action), dtype=int)
    self.permanent_movement  = np.array([0] * len(Action), dtype=int)
    self._board              = board.copy()
    self.board               = board.copy()
    self.grid_size           = board.shape[0]
    self.length              = 0
    self.max_length          = max_length
    self.resurfaced          = -1
    self.directions          = np.full(board.shape, Action.NOTHING.value)
    self.last_torpedo        = np.array([-1, -1], dtype=int)
    self.UNet                = UNet
    self.torpedo_counter     = torpedo_counter
    self.resurfaced_count    = 0

    """
    What can the RL agent see?
    > Agent coordinates: [x, y]
    — Range: [0, 14]

    > Repairable movement: [N, S, E, W]
    — Range: [0, 2]

    > Permanent damage movement: [N, S, E, W]
    — Range: [0, 2]

    > Resurfaced
    - Range: [-1, 8], where -1 means never resurfaced and 0-8 means resurfaced in section 1-9

    > Last torpedo coordinates: (15, 15) grid
    - Range: [0, 10], 
    """
    self.observation_space = Dict({
        # Agent coordinates: [x, y]
        "Coordinates": Box(low=0, high=self.grid_size - 1, shape=(2,), dtype=int),
        # Repairable movement: [N, S, E, W]
        "Repairable_movement": Box(low=0, high=len(Action)-1, shape=(len(Action),), dtype=int),
        # Permanent damage movement: [N, S, E, W]
        "Permanent_movement": Box(low=0, high=len(Action)-1, shape=(len(Action),), dtype=int),
        # Tracks which section resurfaced
        "Resurfaced": Box(low=-1, high=8, shape=(1,), dtype=int),
        # Last torpedo coordinates: [x, y]
        # "Last_torpedo": Box(low=0, high=torpedo_counter.max_fires, shape=(self.grid_size ** 2,), dtype=int),
        # Action masking: prevents illigal moves
        "action_mask": Box(low=0, high=1, shape=(len(Action),), dtype=int)
    })

    """
    What can agent do?
    > Move UP
    > Move DOWN
    > Move LEFT
    > Move RIGHT
    > Move Nothing
    > RESURFACE
    6 actions in total
    """
    self.action_space = Discrete(len(Action))

    # Each action vectorized
    self._vectorize_action = {
        # Going from row 1 to 0 is UP
        Action.UP.value:      np.array([ -1 ,  0 ]),
        # Going from row 0 to 1 is DOWN
        Action.DOWN.value:    np.array([  1 ,  0 ]),
        # Going from column 1 to 0 is LEFT
        Action.LEFT.value:    np.array([  0 , -1 ]),
        # Going from column 0 to 1 is RIGHT
        Action.RIGHT.value:   np.array([  0 ,  1 ]),
        Action.NOTHING.value: np.array([  0 ,  0 ]),
    }

  def update_UNet(self, UNet):
    self.UNet = UNet

  def get_predicted_submarine_location(self):
    # Get the predicted submarine location from the UNet

    prediction = self.UNet(
        torch.tensor(self.board).unsqueeze(0).unsqueeze(0).float(),
        torch.tensor(self._board).unsqueeze(0).unsqueeze(0).float(),
        torch.tensor(sections_to_masks([self.resurfaced])).unsqueeze(0).float()
    )
    prediction_numpy = prediction.cpu().detach().numpy().squeeze().reshape(15, 15)
    index = np.argmax(prediction_numpy)
    x = index % 15
    y = index // 15
    return np.array([y, x])

  def step(self, action):
    """
    RESURFACE action skips all movement related steps
    """
    done = False
    truncated = False
    if action == Action.RESURFACE.value:
      # Calculate which section resurfaced in
      x, y = self.agent[0], self.agent[1]
      section = (y // 5) * 3 + (x // 5)
      self.resurfaced = section

      # No reward
      reward = -1 * (self.resurfaced_count + 1)

      # Punish resurface more and more
      self.resurfaced_count += 1

      # Record which section resurfaced in
      info = {"section": section, "record": True}

      # Reset movement trackers
      self.repairable_movement = np.array([0] * len(Action))
      self.permanent_movement  = np.array([0] * len(Action))

      # Remove previous travelled points on board
      self.board = np.where(self.board == Tile.TRAVELLED.value, Tile.OCEAN.value, self.board)

      # Update length
      self.length += 1

      if self.length >= self.max_length:
        done = True
        truncated = True

      # Calculating section and after resetting movement tracker, get observation space
      observation_space = self.get_observation_space()
      return observation_space, reward, done, truncated, info


    """
    First, change the agent's action based on the algorithm's answer
    Give 1 reward as the longer the agent survives, the better
    """
    x_previous = self.agent[0]
    y_previous = self.agent[1]
    self.board[x_previous, y_previous] = Tile.TRAVELLED.value
    vectorized_movement = self._vectorize_action[action]
    self.agent += vectorized_movement
    reward = 1

    """
    Second, if agent is outside boundaries, in an island or already been there,
    which is not allowed, then reverse action and punish.
    Otherwise, note where agent has been on the board
    """
    true_action = action
    x = self.agent[0]
    y = self.agent[1]
    if x == -1 or y == -1 or x == self.grid_size or y == self.grid_size or self.board[x, y] == Tile.TRAVELLED.value or self.board[x, y] == Tile.ISLAND.value:
      self.agent -= vectorized_movement
      true_action = Action.NOTHING.value
    else:
      self.board[x, y] = Tile.SUBMARINE.value

    if true_action == Action.NOTHING.value:
      reward -= (self.repairable_movement[true_action] + 1)

    """
    Third, update repairable movement in the correct direction.
    If repairable movement is already 3,
    then add to permanent damage movement
    """
    if true_action == Action.NOTHING.value or self.repairable_movement[true_action] < 3:
      self.repairable_movement[true_action] += 1

    elif true_action != Action.NOTHING.value:
      self.permanent_movement[true_action] += 1

    """
    Fourth, add repair logic.
    Check if repairable_movement NORTH, SOUTH or WEST has 3 and EAST is at least 1,
    then substract for NORTH, SOUTH or WEST 3 and 1 from EAST
    """
    if true_action == Action.RIGHT.value and np.any(self.repairable_movement == 3):
      action_to_be_repaired = np.where(self.repairable_movement == 3)[0][0]

      # Only combination of (NORTH, SOUTH, WEST) x (EAST) is repairable
      if action_to_be_repaired != Action.RIGHT.value:
        self.repairable_movement[Action.RIGHT.value] -= 1
        self.repairable_movement[action_to_be_repaired] -= 3

    if true_action != Action.NOTHING.value and true_action != Action.RIGHT.value and self.repairable_movement[true_action] == 3 and self.repairable_movement[Action.RIGHT.value] > 0:
      self.repairable_movement[Action.RIGHT.value] -= 1
      self.repairable_movement[true_action] -= 3

    """
    Fifth, if repairable movement and permanent damage add up to 6 in any direction,
    then simulation is done and punish
    """
    if self.repairable_movement[true_action] + self.permanent_movement[true_action] >= 6:
      done = True

    """
    Sixth, if simulation has been running for X steps,
    then simulation is done and reward with X points
    """
    self.length += 1
    if self.length >= self.max_length:
      done = True
      truncated = True

    """
    Seventh, translate self.agent, self.repairable_movement
    and self.permanent_movement into oberservation_space
    """
    observation_space = self.get_observation_space()
    info = {}

    if true_action != Action.NOTHING.value:
      info["true_action"] = true_action
      info["agent_position"] = self.agent.copy()
      self.directions[x_previous, y_previous] = true_action
    
    if done == True:
      info["record"] = True
      info["directions"] = self.directions

    """
    Eight, torpedo cooldown -1, if on 0, then fire torpedo and reset cooldown
    """ 
    if self.torpedo_counter.can_fire():
      predicted_location = self.get_predicted_submarine_location()
      y, x = predicted_location
      self.last_torpedo[y, x] = self.torpedo_counter.fired

      info["torpedo_fired"] = predicted_location


      distance = np.abs(self.agent - predicted_location).max()
      if distance <= 1:
        print(f"Torpedo hit at {predicted_location} at length {self.length}")
        done = True
        truncated = False
        reward = self.length - self.max_length
        info["torpedo_hit"] = True
      else:
        print(f"Torpedo missed at {predicted_location} with distance {distance} at length {self.length}")

    return observation_space, int(reward), done, truncated, info

  def render(self):
    pass

  def reset(self, manual_spawn=False, random_damage=False, seed=None, options=None):
    super().reset(seed=seed)
    self.np_random = np.random.RandomState(seed)

    # Remove previous been points on board
    self.board = self._board.copy()
    self.directions = np.full(self.board.shape, Action.NOTHING.value)

    # Reset the length
    self.length = 0

    # Reset resurfaced
    self.resurfaced = -1
    self.resurfaced_count = 0

    # Reset torpedo
    self.last_torpedo = np.zeros((self.grid_size, self.grid_size), dtype=int)
    self.torpedo_counter.reset()

    # Initialize random starting position of submarine
    if type(manual_spawn) == bool:
      self.agent = self.np_random.randint(0, self.grid_size, 2)
      while self.board[self.agent[1], self.agent[0]] == Tile.ISLAND.value:
        self.agent = self.np_random.randint(0, self.grid_size, 2)
    else:
      self.agent = manual_spawn
    x = self.agent[0]
    y = self.agent[1]
    self.board[x, y] = Tile.SUBMARINE.value
    self.travelled = [self.agent]

    # Reset movement trackers
    if random_damage:
      self.repairable_movement = self.np_random.randint(0, 3 + 1, 4)
      self.permanent_movement  = self.np_random.randint(0, 3 + 1, 4)
    else:
      self.repairable_movement = np.array([0] * len(Action))
      self.permanent_movement  = np.array([0] * len(Action))

    # Return observation
    return (self.get_observation_space(), {})

  def get_observation_space(self):
    action_mask = []
    current_position = self.agent.copy()
    for a in Action:
      if a == Action.DOWN.value or a == Action.RIGHT.value or a == Action.UP.value or a == Action.LEFT.value:
        vectorized_movement = self._vectorize_action[a.value]
        future_position = current_position + vectorized_movement
        x = future_position[0]
        y = future_position[1]
        if x == -1 or y == -1 or x >= self.grid_size or y >= self.grid_size or self.board[x, y] == Tile.TRAVELLED.value or self.board[x, y] == Tile.ISLAND.value:
          # 0 is Invalid
          action_mask.append(0)
        else:
          # 1 is Valid
          action_mask.append(1)
      else:
        # Resurface and Nothing are always valid
        action_mask.append(1)

    return {
        # Agent coordinates: [x, y]
        "Coordinates": self.agent,
        # Repairable movement: [N, S, E, W]
        "Repairable_movement": self.repairable_movement,
        # Permanent damage movement: [N, S, E, W]
        "Permanent_movement": self.permanent_movement,
        # Tracks which section resurfaced
        "Resurfaced": np.array([self.resurfaced]),
        # Last torpedo coordinates: [x, y]
        # "Last_torpedo": self.last_torpedo.flatten(),
        # Action mask
        "action_mask": np.array(action_mask)
    }


## Agent 2

In [ ]:
class IntermediateCaptainSonarEnv(SimpleCaptainSonarEnv):
  def __init__(self, board, UNet, torpedo_counter, max_length = 50):
    super().__init__(board, UNet, torpedo_counter, max_length)
    self.observation_space["Coordinates"] = Box(low=0, high=1, shape=(self.grid_size ** 2 * len(Tile),), dtype=int)
    self.encoder = OneHotEncoder(sparse_output=False, categories=[range(len(Tile))])

  def get_observation_space(self):
    observation_space = super().get_observation_space()
    encoded_board = self.board.reshape(-1, 1)
    encoded_board = self.encoder.fit_transform(encoded_board).flatten()
    encoded_board = encoded_board.astype(np.int64)
    observation_space["Coordinates"] = encoded_board
    return observation_space

In [ ]:
max_length = 100
trials = 100

torp_counter = Torpedo_Counter(seed=42, low=25, high=50, max_fires=10)
intermediate_env = IntermediateCaptainSonarEnv(board_numpy, initial_UNet_model, torp_counter, max_length)
check_env(intermediate_env)
intermediate_model = PPO("MultiInputPolicy", intermediate_env, verbose=1, learning_rate=1e-4)

In [ ]:
callback_intermediate = ActionLoggerCallback()
CycleModel(intermediate_model, callback_intermediate, "Intermediate", trials)

In [ ]:
callback_intermediate.sequences[:5], callback_intermediate.whole_sequences[:5]

In [ ]:
fig

In [ ]:
test = ""

for _ in range(10):
    test += str(1) + " "

test = test.strip()
print(test.split(" "))

## Agent 4

In [ ]:
# Code has been inspired from https://stable-baselines3.readthedocs.io/en/master/guide/custom_policy.html#custom-feature-extractor

class CustomHybrid(BaseFeaturesExtractor):
    def __init__(self, observation_space: gymnasium.spaces.Dict, linear_dim = 32, feature_extract = 32):
        # We do not know features-dim here before going over all the items,
        # so put something dummy for now. PyTorch requires calling
        # nn.Module.__init__ before adding modules
        super().__init__(observation_space, features_dim=1)

        extractors = {}

        total_concat_size = 0
        # We need to know size of the output of this extractor,
        # so go over all the spaces and compute output feature sizes
        for key, subspace in observation_space.spaces.items():
            if "movement" in key or key == "action_mask":
                # These are discrete
                features = subspace.shape[0]
                extractors[key] = nn.Sequential(
                    nn.Linear(features, linear_dim),
                    nn.ReLU(),
                    nn.Linear(linear_dim, linear_dim),
                    nn.ReLU(),
                    nn.Linear(linear_dim, linear_dim),
                    nn.ReLU(),
                )
                total_concat_size += linear_dim
            elif key == "Coordinates":
                # Run through CNN
                n_input_channels = subspace.shape[0]
                extractors[key] = nn.Sequential(
                  nn.Conv2d(n_input_channels, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.MaxPool2d(2, 2),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.Conv2d(feature_extract, feature_extract, kernel_size=3, stride=1, padding=1),
                  nn.ReLU(),
                  nn.MaxPool2d(2, 2),
                  nn.Flatten(),
                )
                total_concat_size += subspace.shape[1] // 2 // 2 * feature_extract + subspace.shape[1] // 2 // 2 * feature_extract

        self.extractors = nn.ModuleDict(extractors)

        # Update the features dim manually
        self._features_dim = 384

    def forward(self, observations) -> torch.Tensor:
        encoded_tensor_list = []

        # self.extractors contain nn.Modules that do all the processing.
        for key, extractor in self.extractors.items():
            encoded_tensor_list.append(extractor(observations[key]))
        # Return a (B, self._features_dim) PyTorch tensor, where B is batch dimension.
        return torch.cat(encoded_tensor_list, dim=1)

policy_kwargs_hybrid = dict(
    features_extractor_class=CustomHybrid,
    features_extractor_kwargs=dict(linear_dim=32, feature_extract=32),
)

In [ ]:
class ComplexCaptainSonarEnv(SimpleCaptainSonarEnv):
  def __init__(self, board, max_length = 50):
    super().__init__(board, max_length)

    self.board = self.board.astype(np.uint8)
    self.observation_space = Dict({
        "Coordinates": Box(low=0, high=1, shape=(len(Tile), self.grid_size, self.grid_size), dtype=np.uint8),
        "action_mask": self.observation_space["action_mask"]
    })
    self.encoder = OneHotEncoder(sparse_output=False, categories=[range(len(Tile))])

  def get_observation_space(self):
    _observation_space = super().get_observation_space()
    observation_space = {}
    encoded_board = self.board.reshape(-1, 1)
    encoded_board = self.encoder.fit_transform(encoded_board).T
    encoded_board = encoded_board.astype(np.int64)
    observation_space["Coordinates"] = encoded_board.reshape(len(Tile), self.grid_size, self.grid_size)
    observation_space["action_mask"] = _observation_space["action_mask"]
    return observation_space